(benchmarks-overview)=
# How the benchmarks are evaluated

(scorecard-dimensions-at-a-glance)=
## Knowledge-discovery scorecard

`mboostrs-scorecard/2.1` defines twelve dimensions. The table gives the
question each dimension asks. Select its code for the measurement protocol
and the five scoring anchors. The [results scorecard](where_we_stand.ipynb#where-the-scorecard-stands)
shows the actual grades and the evidence gaps for mboostrs.

| Code | Dimension | Question |
| --- | --- | --- |
| [D1a](#scorecard-d1a) | Relationship recovery under a prescribed specification | Given a rough formula, does the library recover the true structure it left open? |
| [D1b](#scorecard-d1b) | Relationship discovery from data alone | Given data and response semantics, does it recover true structure and resist false discoveries? |
| [D2](#scorecard-d2) | Causal humility and support | Do causal claims stay within the design and its assumptions? |
| [D3](#scorecard-d3) | Explanation quality | Is the account of the fitted model faithful, complete, and useful for decisions? |
| [D4](#scorecard-d4) | Importance quantification | How large are the effects, and do effect intervals cover their true values? |
| [D5](#scorecard-d5) | Experimental iteration speed | How long do follow-up fits and diagnostics take? |
| [D6](#scorecard-d6) | Robustness across conditions | How does performance change with sample size, messy inputs, and distribution shift? |
| [D7](#scorecard-d7) | Human-friendliness | Can a person learn the workflow, diagnose a failure, and recover? |
| [D8](#scorecard-d8) | Agent-friendliness | Can an agent complete the analysis through the API and handle errors autonomously? |
| [D9](#scorecard-d9) | Uncertainty honesty (calibration) | Do prediction intervals for new observations cover at their stated rate? |
| [D10](#scorecard-d10) | Reproducibility and auditability | Can a reader reproduce the numbers and audit how the model was chosen? |
| [D11](#scorecard-d11) | Predictive validity | How well does the model predict new observations on the intended split? |

Grades run from 1 to 5 against the anchors below. Half points require a
written justification. **Unmeasured** means the study lacks evidence for a
grade. **Not applicable** means the dimension has no role for that arm in
the comparison, as when the simple reference is itself the D11 baseline.
Studies measure the dimensions their designs support. Historical studies
keep their original rubrics and grades.

Planted-truth data are generated from a known model, so discovery can be
checked against the known effects. Real-data studies use held-out losses,
prespecified reference models, or published effect estimates. Each of these
answers a different question. Studies also differ in tuning budgets, splits,
and the package versions used. Read grades within their named study.

(running-and-updating-benchmarks)=
## How benchmarks are structured and rerun

Part V is organized into four tiers so headline evidence stays separated from internal ablations:

1. **Headline competitive benchmarks** — cross-library studies that determine where mboostrs stands overall (`heldout_competition`, `distributional_shootout`, `competitive_shootout`, and the public domain suites).
2. **Discovery, causal, and agent workflows** — end-to-end discovery, causal adjustment, forecasting, and agent-workflow evaluations.
3. **Uncertainty and calibration studies** — focused audits of interval coverage, conformal scoring, and response-scale invariance.
4. **Component ablations and engineering archive** — stopping-rule, encoding, interaction, and speed studies that document specific design choices.

Every maintained runner, output artifact, and dependent book page is registered in `benches/registry.json` and executed through `benches/run.py`:

```bash
.venv/bin/python benches/run.py list --tier headline
.venv/bin/python benches/run.py status
.venv/bin/python benches/run.py run distributional_comparison --update-book
.venv/bin/python benches/run.py run realdata_autospec --workers 8 --update-book
```

Studies marked `mboostrs_only` remeasure only the mboostrs arms on identical draws and merge them into the stored artifact while retaining competitor rows verbatim, keeping iteration fast as the library improves.

## Terms

A *data-generating process* (DGP) is the recipe that produced the data,
including which variables truly matter and how. The *true positive rate*
(TPR) is the share of planted effects the method found. The *false-discovery
proportion* is the share of reported findings that were false. The
*false-discovery rate* (FDR) is its expectation across repetitions.
Benchmarks summarize false discoveries over the recorded tasks and seeds.
The *information contract* specifies what the analyst receives before
fitting, such as a family, formula, or candidate predictors. Recovery scores
count the structure the analyst was asked to find.

(information-contracts-511)=
## What the analyst is given

How much the analyst knows in advance changes what discovery means, so
recovery is scored under fixed information contracts.

Under **Track A**, the analyst is handed a rough formula: the family and link,
the candidate predictors and the kinds of term to consider. The true structure
is not supplied. Only what the formula left *open* is scored; a term the formula
already named earns no discovery credit. Under **Track B**, the analyst gets the
data, the meaning of the response, the metadata, the design and a budget, but no
formula at all. Every piece of recovered structure is scored.

A middle track, **partial knowledge**, supplies some components and leaves others
for the analyst to find. Only the latter count toward recovery; the protocol
must list the supplied components.

The comparison reference depends on whether the generating model is known. On
planted-truth data the known generating model (the *oracle*) gives the best
attainable loss, so a library's loss can be compared with it. On real data the
generating model is unknown, so the comparison is a *prespecified* reference
specification, fixed before any test loss is seen. Tuning and screening choose
models by predictive loss and carry no hypothesis-test error guarantee. Where
losses are reported, the scorecard prefers paired per-task differences over
pooled averages: the *excess* of a library's loss over the reference loss, and
the *gap* between a library's loss and the oracle loss. It asks for uncertainty
across tasks and seeds.

## Definitions and rubrics

(scorecard-d1a)=
### D1a. Relationship recovery under a prescribed specification

**What it covers.** D1a measures whether a library finds the structure that a
rough formula left open. The analyst supplies the family, the candidate
predictors and the term types, but not the true shape.

**Why it matters.** Real analyses rarely start from nothing. A statistician
brings domain knowledge, writes a tentative model, then asks what it missed. A library that fits only the named terms leaves the analyst to search for omitted
structure. D1a rewards a library that searches the open part and controls its
false discoveries while doing so.

**Measurement.** Freeze the prescribed formula (family/link, candidate predictors, term types). Planted-truth suite adds structure the formula omits (interactions, nonlinearities, heteroscedasticity, pure noise). Report TPR and FDR per structure type for the *open* terms only. List the prescribed terms and award no discovery credit for recovering them.

| Score | Anchor |
|---|---|
| 1 | Fits the prescribed formula literally; no search over the open structure |
| 2 | Ad-hoc residual scanning (variable importance) with no false-discovery control |
| 3 | Structured search over the open terms with explicit evidence and decision rules; error-control claims state their assumptions |
| 4 | Validated on planted-truth benchmarks under a frozen prescribed formula; false-discovery behavior on the open structure characterized |
| 5 | All of 4, plus higher-order open structure (interactions, distributional effects) with published recovery rates |

**Illustration.** Prescribe a smooth additive model for wages. Plant a true
education-by-experience interaction that the formula never names. A level-1
library returns the smooth terms and stops; a level-4 library flags the
interaction and reports how often its open-structure claims are wrong.

(scorecard-d1b)=
### D1b. Relationship discovery from data alone

**What it covers.** D1b measures finding true structure with no formula at all.

**Why it matters.** Without a supplied formula, the analyst must identify candidate effects as well
as estimate them. Scoring false discoveries alone would reward a method that
always reports nothing, so D1b also measures recovery of planted structure.

**Measurement.** Supply no formula: data, response semantics, metadata, design and budget only. Score TPR and FDR by structure type against planted truth. A library that reports nothing cannot outscore one that reports true structure, and must also show zero false structure on pure-noise tasks. Report the discovery budget and the search scope.

| Score | Anchor |
|---|---|
| 1 | No systematic discovery; relationships found only by manual specification |
| 2 | Ad-hoc aids (variable importance) with no false-discovery control |
| 3 | Structured screening with explicit evidence and decision rules; any error-control claims state their assumptions |
| 4 | Screening + adoption validated on planted-truth benchmarks; false-discovery behavior characterized |
| 5 | All of 4, plus higher-order structure (interactions, distributional effects) with published recovery rates |

**Illustration.** On a table with no signal, the target is no reported effects. A level-3 library returns a small screened set with stated
rules and reports none of it when the noise test comes up empty; a level-2
importance plot always highlights some variable and so reports a finding every time.

(scorecard-d2)=
### D2. Causal humility and support

**What it covers.** D2 measures whether a library stops an analyst from reading
causes out of a predictive fit, and supports causal estimation with stated identification assumptions.

**Why it matters.** A model can predict well while its top drivers are
confounded. Treating a predictive ranking as a cause leads to bad decisions,
especially when the analyst is told to trust the top of the list.

**Measurement.** Documentation audit + workflow checklist.

| Score | Anchor |
|---|---|
| 1 | Encourages causal reading of predictive features ("top drivers," no caveats) |
| 2 | Neutral — neither encourages nor discourages causal interpretation |
| 3 | Docs scope findings as associational; supports confounder-adjustment workflows |
| 4 | All of 3, plus built-in sensitivity/robustness checks for key findings |
| 5 | All of 4, plus experimental-design support integrated with modeling |

**Worked example.** A fit reports that a job-training program drives earnings.
The data also show that participants were older on average, and age raises
earnings. A predictive-only library prints training at the top of the driver
list with no caveat: that is the level-1 behaviour. A level-4 workflow identifies the association as observational, adjusts for age
and prior earnings, and checks sensitivity to an unmeasured confounder. In this
example it reports a smaller effect with an uncertainty interval and states the
assumptions needed for a causal interpretation.

**Nearby dimensions.** D2 does not grade whether structure was found (D1a and
D1b) or whether predictions generalise (D11). Correct discovery followed by a
causal misreading is a D2 failure alone.

(scorecard-d3)=
### D3. Explanation quality

**What it covers.** D3 measures how well a library turns findings into a
faithful, complete and decision-useful narrative.

**Why it matters.** Analysts need to explain which effects the model supports and which limitations
affect a decision. D3 grades the report produced from the fitted model.

**Measurement.** Human-grounded task: analysts/agents write reports from library outputs on a planted-truth dataset; grade faithfulness and completeness against the key. Record whether the run was Track A (prescribed specification) or Track B (data alone) and report explanation credit for discovered versus prescribed terms separately.

| Score | Anchor |
|---|---|
| 1 | No explanation outputs; analyst invents narrative from raw coefficients |
| 2 | Per-prediction explanations only (SHAP-style); no dataset-level story |
| 3 | Dataset-level summaries (effect plots, importance rankings) faithful to the fitted model |
| 4 | Ranked, evidence-graded findings with uncertainty — reportable nearly verbatim |
| 5 | Findings compose into a coherent report: what we know / don't know / check next |

**Contrast with D4.** D3 owns the narrative and its faithfulness; D4 owns the
effect size and its uncertainty. Two libraries can see the same wage regression
and diverge. Library A returns a fluent, well-ordered summary of which
predictors matter, but every number is a point value with no interval: this can score high on D3 and low on D4. Library B returns a
raw coefficient table where each effect carries a confidence interval that
covers the truth about as often as it claims, but there is no story, no ranking
and no plain-language summary: high D4 and low D3.

(scorecard-d4)=
### D4. Importance quantification

**What it covers.** D4 measures whether a library reports effect sizes with
uncertainty intervals whose coverage can be assessed, rather than only rankings
or heuristic scores.

**Why it matters.** An importance ranking orders predictors but gives neither an effect magnitude
nor its uncertainty. A decision needs the size of the change in the outcome
when a predictor changes, with an interval for that size.

**Measurement.** Coverage calibration on planted-truth DGPs; interval width at matched coverage. Under the Track A contract, report intervals for discovered (open) effects separately from prescribed effects.

| Score | Anchor |
|---|---|
| 1 | Point estimates only; no uncertainty |
| 2 | Heuristic scores (gain, permutation) with no sampling interpretation |
| 3 | Confidence/credible intervals under stated assumptions |
| 4 | Coverage validated empirically (90% interval covers ~90%) |
| 5 | Decision-relevant magnitudes ("changes prediction by X ± Y on the outcome scale") |

**Illustration.** One library reports that education raises wages by 1.2 log
points with a 90% interval of 0.9 to 1.5, and those intervals contain the truth
90% of the time; that is level 4. Another reports that education is important,
with a gain score of 0.42 and no scale or error; that is level 2.

**Nearby dimensions.** D4 and D9 both concern intervals but not the same ones.
D4 covers effect intervals: a coefficient, an average treatment effect, an
effect curve. D9 covers prediction intervals: a new observation. A library can
calibrate one and not the other. D4 and D11 also differ: D4 asks whether the
effect estimate and its interval are accurate; D11 asks whether predictions
generalise.

(scorecard-d5)=
### D5. Experimental iteration speed

**What it covers.** D5 measures the wall-clock time from the question, what if
we tried X, to having an answer. D5 always means iteration speed.

**Why it matters.** Fit and diagnostic times determine how many alternative specifications an
analyst can evaluate in a session.

**Measurement.** Standardized "analyst session": 10 specified follow-up fits on 50k×20 data, incl. diagnostics. Report median and p90 wall-clock.

| Score | Anchor |
|---|---|
| 1 | Single fits take minutes+ on medium data; iteration impractical |
| 2 | Tens of seconds per fit; a handful of iterations per session |
| 3 | Seconds per fit; dozens of iterations; cheap diagnostics |
| 4 | Sub-second refits; batched counterfactuals; interactive loop |
| 5 | Incremental/online updating — follow-ups don't require full refits |

**Illustration.** Consider ten refits, each changing a spline knot and inspecting the effect.
At forty seconds per refit, the fits take almost seven minutes; at four tenths
of a second, they take a few seconds.

**Nearby dimension.** D5 is raw latency. D8 is whether an autonomous agent can
drive the loop at all through the API. A fast library with untyped,
unparseable outputs can be quick and still agent-hostile, and a slower library
with documented inputs, outputs and errors can support autonomous analysis.

(scorecard-d6)=
### D6. Robustness across conditions

**What it covers.** D6 measures graceful behaviour on small samples, large
samples, messy data and shifted data.

**Why it matters.** Real tables arrive with missing values, outliers and mixed
types, and they grow or shrink. Requiring complete numeric inputs leaves the analyst to handle missing values
and categorical predictors before fitting.

**Measurement.** Fixed battery: small-n (n=200), messy (15% MCAR + outliers + mixed types), large-n (5M rows). Report D1a and D1b recovery + runtime per condition, under both information contracts where feasible.

| Score | Anchor |
|---|---|
| 1 | Requires clean complete numeric matrices; fails/silently misbehaves otherwise |
| 2 | Basic missingness/mixed-type handling, documented |
| 3 | Discovery quality measured across small-n and messy benchmarks; graceful, documented degradation |
| 4 | All of 3, plus large-n scalability demonstrated |
| 5 | All of 4, plus shift/stress diagnostics that warn when findings may not transfer |

**Illustration.** Delete 15% of the cells at random and add 1% outliers. A
level-3 library keeps most of its recovery and records the drop; a level-1
library errors on the first missing cell.

(scorecard-d7)=
### D7. Human-friendliness

**What it covers.** D7 measures learnability, diagnosability and trust for a
competent human analyst.

**Why it matters.** The analyst is part of the discovery loop. Cryptic errors,
misleading defaults and unguided workflows increase the time needed to complete and diagnose an analysis.

**Measurement.** Task-based study: time-to-first-discovery, error-recovery rate (or issue-tracker mining as proxy). Run the task under both the Track A prescribed-specification and Track B data-alone contracts and report separately, since guidance needs differ.

| Score | Anchor |
|---|---|
| 1 | Cryptic API, undocumented failure modes, misleading defaults |
| 2 | Usable with effort; discovery workflows unguided |
| 3 | Coherent API, guided workflows, informative errors |
| 4 | Executable case studies for main discovery patterns; diagnostics point at fixes |
| 5 | Evidence the docs work (user studies or issue-tracker analysis) |

**Illustration.** A mistyped family raises a message that names the problem,
suggests the intended family and links its page, which is level 3, against a
stack trace that ends somewhere inside a compiled backend, which is level 1.

(scorecard-d8)=
### D8. Agent-friendliness

**What it covers.** D8 measures whether an autonomous agent can drive the whole
discovery loop through the API.

**Why it matters.** In the fully automated setting the analyst is a program. Structured findings and errors that name a remedy let the agent choose its next
API call without reconstructing those details from a plot or prose.

**Measurement.** Frozen-prompt LLM agent performs discovery on planted-truth datasets using only the library; grade D1a/D1b recovery and D3 report quality. Report Track A and Track B runs separately, with the prescribed terms identified for Track A.

| Score | Anchor |
|---|---|
| 1 | Every step needs human judgment ("look at the plot and decide") |
| 2 | Programmatic API, but unstructured outputs or uninformative errors |
| 3 | Structured outputs (typed findings, machine-readable diagnostics); actionable error codes |
| 4 | Documented agent contract (assumptions, checks, preflight/validation hooks) |
| 5 | Demonstrated: agent completes discovery benchmarks end-to-end, zero human assists |

**Illustration.** A diagnostic call returns a typed table of findings with a
severity and a suggested next call, versus a call that returns a free-text
paragraph the agent must parse and guess at.

(scorecard-d9)=
### D9. Uncertainty honesty (calibration)

**What it covers.** D9 measures whether stated uncertainty matches reality and
widens when the data warrant. D9 always means calibration.

**Why it matters.** An interval advertised at 90% coverage can lead an analyst to underestimate
prediction error if it covers substantially fewer than 90% of new outcomes.

**Measurement.** Coverage experiments on planted-truth DGPs incl. heteroscedastic and small-n variants; empirical vs. nominal coverage. Attribute coverage to the track that produced the estimate (D1a prescribed vs D1b discovered) so calibration is not credited to supplied structure.

| Score | Anchor |
|---|---|
| 1 | No uncertainty quantification |
| 2 | Uncertainty reported but uncalibrated |
| 3 | Coverage validated on standard DGPs; approximately nominal |
| 4 | Adaptive: widens under heteroscedasticity, small-n, extrapolation — demonstrated |
| 5 | Diagnostics warn when the uncertainty itself is unreliable |

**Illustration.** Under a heteroscedastic DGP, a level-3 library's bands are
nominal on average but flat, so they under-cover in the high-variance region; a
level-4 library lets the band widen there.

(scorecard-d10)=
### D10. Reproducibility and auditability

**What it covers.** D10 measures whether a third party can reproduce every
number and audit every decision.

**Why it matters.** Reproducing a result requires the data, settings, seeds and code that produced
it. Records of rejected candidates are also needed to inspect model selection.

**Measurement.** Independent reproduction from the report alone; grade numerical agreement and trail completeness.

| Score | Anchor |
|---|---|
| 1 | Undocumented randomness or hidden state |
| 2 | Seeds fixable, but decision trail not recorded |
| 3 | Full decision log (candidates, adoptions/rejections with reasons, seeds) |
| 4 | One-command reproduction of the entire discovery report |
| 5 | Artifact versioning (data, code, environment hashes alongside findings) |

**Illustration.** A second analyst follows the instructions in the report and recovers the same
effects to the last digit, and can see from the log which candidates were
considered and why each was dropped; that is level 4, against a script that
prints a number while hiding its seed and its search.

**Nearby dimension.** D3 is whether the story is faithful for a reader; D10 is
whether an independent party can reproduce the numbers. A persuasive report with
no seed or decision trail can be high D3 and low D10.

(scorecard-d11)=
### D11. Predictive validity

**What it covers.** D11 measures useful predictions on observations that were
not used to construct or select the model.

**Why it matters.** Leakage and model selection on test responses can lower the reported loss
without improving prediction on new observations. D11 therefore requires data
excluded from preprocessing, fitting and selection.

**Measurement.**

- Fix in advance the estimand (the quantity being predicted, such as a class
  probability or a conditional mean), the information available at prediction
  time, the loss, and the reference population.
- Evaluate on outer test data that no fitting or selection step has used, or
  with nested validation (an inner resampling loop for every choice, inside an
  outer loop that only scores).
- Learn all preprocessing, screening and tuning on the permitted training data.
- Use splits appropriate to dependence, timing and deployment.
- Report per-task loss, paired differences with uncertainty, how comparators
  were chosen and their budgets, sample sizes, failures, and every earlier use
  of the test data.
- On real data, compare with the prespecified reference through the paired
  excess; the gap to the oracle (the known generating model) exists only for planted-truth data (see "What the
  analyst is given").
- Keep classification, mean prediction, distribution prediction and
  causal-effect prediction separate; their scores are not pooled.
- Grade prediction-interval coverage under D9 and effect-interval coverage
  under D4. A study that grades them under D11 states so.

| Score | Anchor |
|---|---|
| 1 | Measured leakage, invalid prediction timing, or failure to beat a prespecified simple baseline |
| 2 | Some predictive utility, but evaluation or comparison is too limited to support generalization claims |
| 3 | Honest outer validation shows useful improvement over a prespecified baseline, with uncertainty and failures reported |
| 4 | Competitive predictive performance demonstrated across prespecified task families under matched tuning and compute budgets |
| 5 | Level 4 confirmed on held-back task families and relevant deployment splits, with material losses and transfer limits characterized |

(what-goes-where)=
## Dimension assignment by result type

A study can use the same data to answer several questions. The
following assignments keep the quantity being measured explicit.

| Result | Dimension and scope |
| --- | --- |
| Loss on an untouched test split | D11 measures prediction for the stated outcome and deployment split. |
| Recovery of a planted interaction and false selections among noise terms | D1a when the supplied formula left those terms open. D1b when the analyst received data without a formula. |
| A faithful explanation supported by the fit | D3 assesses the account of the model. |
| An effect estimate and its interval | D4 assesses effect magnitude and coverage of true effects. |
| Coverage for future observations | D9 assesses prediction intervals at their stated coverage. |
| A treatment effect with a sensitivity analysis | D2 assesses the identification and sensitivity reasoning. D4 assesses the magnitude and interval. |
| Wall-clock time for ten specified follow-ups | D5 assesses iteration speed at a fixed work boundary. |
| An agent completes a frozen task with zero human assists | D8 measures autonomous completion. D7 needs evidence from people. |
| An independent analyst reproduces the recorded numbers | D10 assesses reproducibility and the audit trail. |
| Recovery across small, large, or messy samples | D6 measures performance across conditions. Recovery within each condition provides D1a or D1b evidence under its information contract. |

(compact-cheat-sheet)=
## Aggregate scores and weights

A weighted aggregate is permitted only when all twelve dimensions are
scored on one prespecified battery in one operating mode. The two operating
modes are *fully automated* discovery, in which an AI agent selects the model,
writes the formula, diagnoses problems and reports findings without human
input, and *semi-automated* discovery, in which a human analyst uses the
library to test hypotheses and interpret findings. The mode weights are:

| Dimension | Fully automated | Semi-automated |
|---|---|---|
| D1a Relationship recovery (prescribed) | 2× | 2× |
| D1b Relationship discovery (data alone) | 2× | 2× |
| D2 Causal humility | 1× | 1× |
| D3 Explanation / story | 2× | 1× |
| D4 Importance quantification | 1× | 2× |
| D5 Iteration speed | 2× | 1× |
| D6 Robustness | 1× | 1× |
| D7 Human-friendliness | 0.5× | 2× |
| D8 Agent-friendliness | 2× | 0.5× |
| D9 Uncertainty honesty | 2× | 1× |
| D10 Reproducibility | 1× | 1× |
| D11 Predictive validity | 2× | 2× |

Part V reports individual dimensions. None of these studies measures all
twelve on the same battery and operating mode. The
[results scorecard](where_we_stand.ipynb#where-the-scorecard-stands) keeps
numerical grades with their protocol, and the
[benchmark records](../appendix/benchmark_records.md) expose protocols,
deviations and run reports for audit.

## Comparison studies in Part III

Part III contains comparison studies that score fitted models against known generating functions:

- [](../cases/library_comparison.ipynb) and [](../cases/library_comparison_interactions.ipynb)
- [](../cases/distributional_comparison.ipynb)
- [](../cases/benchmark_insight.ipynb), [](../cases/benchmark_causal.ipynb), [](../cases/benchmark_runtime_api.ipynb)
- [](../cases/benchmark_structure.ipynb), [](../cases/benchmark_uncertainty.ipynb), [](../cases/benchmark_survival.ipynb)
- [](../cases/benchmark_classes.ipynb), [](../cases/benchmark_spatial.ipynb), [](../cases/benchmark_drivers.ipynb)
- [](../cases/benchmark_shapes.ipynb), [](../cases/benchmark_pairs.ipynb), [](../cases/benchmark_realdata.ipynb)
- [](../cases/benchmark_lss_drivers.ipynb) and [](../cases/benchmark_preexploration.ipynb)

## Public data sets against other libraries

Four pages score mboostrs against statsmodels, pyGAM, scikit-learn, LightGBM,
XGBoost and CatBoost on public data sets with a published analysis, one
modelling problem each, and a fifth measures fit time and peak memory. Every method sees the same random splits. The pages
report held-out scores with their split-to-split uncertainty, the time and peak
memory of a fit, the warnings and failures each method raised, and the code each
one took. The accompanying analyses relate the score differences to the measured response,
predictors and model specifications.

- [](public_count.ipynb) — over-dispersed counts (Long's bioChemists): what the family buys against what the mean model buys.
- [](public_binary.ipynb) — labour-force participation (Mroz): an imputed covariate that rewards flexibility for the wrong reason.
- [](public_spatial.ipynb) — heavy metals on the Meuse floodplain: covariates, a spatial surface, kriging and trees on 155 points.
- [](public_distributional.ipynb) — wages and their spread (CPS 1988): mean-only models against models of the variance.
- [](public_scaling.ipynb) — fit time and peak memory as the number of rows grows, at a fixed amount of boosting.

All five read artifacts written by the runner `benches/bench_public_tasks.py`,
which is not included in the repository. The artifacts record the runner's
sha256, the data digests, seeds, thread counts and library versions.


## Agent analysts on a planted-truth problem

[](claims_discovery.ipynb) gives seven libraries (mboostrs, XGBoost, LightGBM,
CatBoost, InterpretML EBM, pyGAM, statsmodels) to seven Claude Haiku analysts,
one each, with the same written brief. The data are the real California census
covariates with a simulated claim-cost outcome, so every discovery claim, effect
interval and prediction interval is scored against the truth. Each analyst's
pipeline is then applied, unchanged, to twelve hidden draws, including two with no signal
and two with 800 rows, and the results are graded on the knowledge-discovery
scorecard. InterpretML EBM is the Explainable Boosting Machine.

[](causal_discovery.ipynb) gives five libraries (mboostrs, EconML, DoubleML,
statsmodels, LightGBM) to five Claude Haiku analysts with a written brief for a
program evaluation: a non-randomized job-training program, follow-up earnings
missing for 16% of applicants, and thin overlap above an eligibility limit. The
applicants are real 1988 survey records; enrolment and outcomes are simulated
from a known model, so average effects, per-applicant effects, causal roles and
claims on data with no effect are scored against the truth.

[](pm25_forecast_competition.ipynb) compares seven libraries on hourly Beijing
PM2.5 forecasting under a time-shifted future-year split (2014 held out),
measuring log-scale RMSE, multi-quantile pinball loss, and 80% interval
coverage under seasonal shift.

[](blind_agent_five_libraries.md) compares mboostrs against CatBoost, LightGBM,
InterpretML EBM, and NGBoost under a Track B blind-agent protocol on an unseen
heteroscedastic dataset with missing covariates, a correlated decoy, noise
columns, and an unseen test category, reporting point RMSE, latent-mean
recovery, Gaussian NLL, spread correlation, and 90% interval calibration.

## Predictive validity in two tracks

[](predictive_tracks.ipynb) compares two ways of using mboostrs, fitting a
supplied structure and discovering the structure from the data, with four
tree libraries tuned on validation data. The draws are independent and have
smooth, interaction and factor-curve structure. The page reports paired loss
gaps with their standard errors. It grades D11 only.

## Predictor encoding

[](encoding_tracks.ipynb) encodes the same eight simulated draws in five ways
and compares mboostrs with four tuned tree libraries and ridge regression in
both information tracks (a supplied true structure, and discovery from the data
alone). Five encodings in two tracks give ten cells. With a ceiling of 1,600
boosting updates, the mboostrs test MSE is at most 1% above that of CatBoost,
the best tree library, in eight of the ten cells. The page grades D11 only.

## Speed and scoring on four hard panels

The [speed and scoring study](d5_gap_closing_round3.ipynb) covers four panels selected for slow or
inaccurate mboostrs results: Poisson screening on the quine counts, quantile
binning for large tables (3.3 times faster than the unbinned fit, and faster
than LightGBM in the same comparison), probability scoring on the cancer panel, and the representation
of a joint boundary.